# Spotify Tracks — Exploratory Data Analysis
End-to-end EDA to understand the dataset before building the Streamlit BI dashboard.

In [ ]:
## 1. Install Required Libraries
# Run this cell once to install all dashboard dependencies.
# %pip install streamlit pandas plotly duckdb kaggle seaborn matplotlib

In [ ]:
## 2. Imports & Data Loading
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import duckdb

sns.set_theme(style="darkgrid", palette="muted")
pd.set_option("display.max_columns", None)

DATA_PATH = os.path.join("..", "data", "spotify_tracks.csv")
df = pd.read_csv(DATA_PATH, index_col=0)
df["duration_min"] = df["duration_ms"] / 60_000
df["explicit"] = df["explicit"].astype(bool)

print(f"Shape: {df.shape}")
df.head()

In [ ]:
## 3. Schema & Missing Values
print("=== dtypes ===")
print(df.dtypes)
print("\n=== Missing values ===")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\n=== Basic statistics ===")
df.describe(include="all")

In [ ]:
## 4. Popularity Distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df["popularity"], bins=50, color="#1DB954", edgecolor="black")
ax.set_title("Popularity Distribution")
ax.set_xlabel("Popularity Score (0–100)")
ax.set_ylabel("Track Count")
plt.tight_layout()
plt.show()

In [ ]:
## 5. Top Genres by Track Count
genre_counts = df["track_genre"].value_counts().head(20).reset_index()
genre_counts.columns = ["genre", "count"]
fig = px.bar(
    genre_counts, x="count", y="genre", orientation="h",
    title="Top 20 Genres by Track Count",
    color="count", color_continuous_scale="Greens",
    labels={"count": "Track Count", "genre": "Genre"},
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
## 6. Correlation Heatmap of Audio Features
AUDIO_FEATURES = [
    "danceability", "energy", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo",
    "loudness", "popularity", "duration_min",
]
corr = df[AUDIO_FEATURES].corr()
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn", center=0, ax=ax)
ax.set_title("Audio Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
## 7. Energy vs Danceability (scatter, coloured by Valence)
sample = df.sample(5000, random_state=42)
fig = px.scatter(
    sample, x="energy", y="danceability", color="valence",
    color_continuous_scale="RdYlGn",
    hover_data=["track_name", "artists", "track_genre"],
    opacity=0.6,
    title="Energy vs Danceability (coloured by Valence, sample of 5 000)",
    labels={"energy": "Energy", "danceability": "Danceability", "valence": "Valence"},
)
fig.show()

In [ ]:
## 8. Top 10 Artists by Average Popularity
top_artists = (
    df.groupby("artists")["popularity"]
    .agg(["mean", "count"])
    .query("count >= 5")
    .rename(columns={"mean": "avg_popularity", "count": "track_count"})
    .sort_values("avg_popularity", ascending=False)
    .head(10)
    .reset_index()
)
fig = px.bar(
    top_artists, x="avg_popularity", y="artists", orientation="h",
    title="Top 10 Artists by Avg Popularity (min 5 tracks)",
    color="avg_popularity", color_continuous_scale="Greens",
    text="track_count",
    labels={"avg_popularity": "Avg Popularity", "artists": "Artist", "track_count": "Tracks"},
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
## 9. DuckDB Quick Queries
con = duckdb.connect()
con.register("spotify", df)

# Average audio features per genre
genre_summary = con.execute("""
    SELECT
        track_genre,
        COUNT(*)                      AS track_count,
        ROUND(AVG(popularity), 2)     AS avg_popularity,
        ROUND(AVG(danceability), 4)   AS avg_danceability,
        ROUND(AVG(energy), 4)         AS avg_energy,
        ROUND(AVG(valence), 4)        AS avg_valence
    FROM spotify
    GROUP BY track_genre
    ORDER BY avg_popularity DESC
    LIMIT 15
""").fetchdf()

genre_summary